# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



In [2]:
#load data (model)

res_savename = 'llm_response_impact_labelled_reports_test_continue_41rep_meta-llama_llama-4-scout-17b-16e-instruct.csv'
response_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

#load data (labelled)
#res_savename = "labelled_reports_impacts_all.csv"
#response_df = pd.read_csv(DATA_LABELLED+res_savename)


In [3]:
#load data (labelled)
#fnla = "labelled_8reports_impact_laura.csv"
#labelled_laura = pd.read_csv(DATA_LABELLED+fnla)
#fnlu = "labelled_reports_impacts_luca.csv"
#labelled_luca = pd.read_csv(DATA_LABELLED+fnlu)#.drop(["Unnamed: 0"],axis=1)
#
##reformat to be consistent
#labelled_laura["reportDate"] = pd.to_datetime(labelled_laura["reportDate"], dayfirst=True) #reformat date to be consistent
#labelled_luca["reportDate"] = pd.to_datetime(labelled_luca["reportDate"]) #reformat date to be consistent
#labelled_luca.rename({"annotation":"impactsAnnotation", "impactSubType":"impactSubtype"},inplace=True, axis=1)
#labelled_laura.rename({"annotation":"impactsAnnotation", "impactSubType":"impactSubtype"},inplace=True, axis=1)
#
#response_df = pd.concat([labelled_laura, labelled_luca]).reset_index(drop=True)
#res_savename = "labelled_reports_impacts_all.csv"
#response_df.to_csv(DATA_LABELLED+res_savename, index=False)


In [4]:
response_df[response_df["appealCode"]=="MDRUG050"]

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,hazards,impactsAnnotation,impactValueMin,impactValueMax,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
315,Affected People,69283.0,people,exact,['Uganda'],"['Central region', 'Eastern region', 'Western ...",2024.0,4.0,1.0,2024.0,...,"['Flood', 'Hailstorm', 'Landslide']","['Floods have affected 69,283 people since Apr...",69283.0,69283.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
316,Displaced People,10366.0,families,exact,['Uganda'],"['Central region', 'Eastern region', 'Western ...",2024.0,4.0,1.0,2024.0,...,"['Flood', 'Hailstorm', 'Landslide']","['10,366 families displaced due to floods and ...",10366.0,10366.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
317,Injured People,166.0,people,exact,['Uganda'],['Butaleja district'],2024.0,4.0,3.0,NaN,...,['Flood'],['166 people injured due to floods in Butaleja...,166.0,166.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
318,Human Health and Wellbeing,905.0,people,exact,['Uganda'],['Sironko district'],2024.0,4.0,23.0,NaN,...,['Hailstorm'],['905 people left homeless due to hailstorm in...,905.0,905.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
319,Residential Buildings,172.0,houses,exact,['Uganda'],['Bulambuli district'],2024.0,4.0,3.0,NaN,...,['Flood'],['172 houses destroyed due to floods in Bulamb...,172.0,172.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
320,Residential Buildings,150.0,houses,exact,['Uganda'],['Butaleja district'],2024.0,4.0,3.0,NaN,...,['Flood'],['150 houses completely destroyed in Butaleja ...,150.0,150.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
321,Residential Buildings,108.0,houses,exact,['Uganda'],['Butaleja district'],2024.0,4.0,3.0,NaN,...,['Flood'],['108 houses partially damaged in Butaleja dis...,108.0,108.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
322,Residential Buildings,46.0,houses,exact,['Uganda'],['Namisindwa district'],2024.0,4.0,2.0,NaN,...,['Hailstorm'],['46 houses completely destroyed in Namisindwa...,46.0,46.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
323,Residential Buildings,104.0,houses,exact,['Uganda'],['Namisindwa district'],2024.0,4.0,2.0,NaN,...,['Hailstorm'],['104 houses partially damaged in Namisindwa d...,104.0,104.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...
324,Residential Buildings,82.0,houses,exact,['Uganda'],['Mbale district'],2024.0,4.0,NaN,NaN,...,['Hailstorm'],['82 houses completely destroyed in Mbale dist...,82.0,82.0,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['appeal: mdrug050total dref allocation: chf 4...


In [5]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df.columns else response_df

In [6]:
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax"]#"startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"
list_cols = ["location", "hazards", "impactsAnnotation"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [7]:
#process impactValue
def parse_impact_value_precision(x):
    """parse impact values givent precision levels.
    Ensures that min are mins, max maxs.
    """
    all_values = np.array([x["impactValueMin"], x["impactValueMax"], x["impactValue"]])
    all_values[all_values is None] = np.nan
    min_value = np.nanmin(all_values)
    max_value = np.nanmax(all_values)
    imp_value_min = min_value if not np.isnan(x["impactValueMin"]) else x["impactValueMin"]
    imp_value_max = max_value if not np.isnan(x["impactValueMax"]) else x["impactValueMax"]
    imp_value = imp_value_max if not np.isnan(imp_value_max) else max_value
    #reorder
    #imp_valueMin = x["impactValueMin"] if not np.isnan(x["impactValueMin"]) else None
    #imp_valueMax = x["impactValueMax"] if not np.isnan(x["impactValueMax"]) else None
    #imp_value = x["impactValue"] if not np.isnan(x["impactValueMax"]) else None
    return pd.Series([imp_value, imp_value_min, imp_value_max], index=["impactValue", "impactValueMin", "impactValueMax"])

response_df_proc[["impactValue", "impactValueMin", "impactValueMax"]] = response_df_proc.apply(parse_impact_value_precision, axis=1)

/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_64239/3705022022.py:8: RuntimeWarning: All-NaN slice encountered
  min_value = np.nanmin(all_values)
/var/folders/y5/t1z41tgj7dv50sm2_29dn8740000gp/T/ipykernel_64239/3705022022.py:9: RuntimeWarning: All-NaN slice encountered
  max_value = np.nanmax(all_values)


In [8]:
response_df_proc[["impactValue", "impactValueMin", "impactValueMax"]]

,impactValue,impactValueMin,impactValueMax
0,247408.0,247408.0,247408.0
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,68.0,NaN,NaN
...,...,...,...
340,NaN,NaN,NaN
341,NaN,NaN,NaN
342,NaN,NaN,NaN
343,NaN,NaN,NaN


In [9]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [10]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [11]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

In [ ]:
#reclassify impacType
impact_kw_reclass = {
    'Affected People': r"\bAffected People\b",
    'Injured People': r"\bInjured People\b",
    'Displaced People': r"\bDisplaced People\b",
    'Homeless People': r"\bHomeless People\b",
    'Missing People': r"\bMissing People\b",
    'Human Deaths': r"\bHuman Deaths\b",
    'Human Health and Wellbeing': r"\bHuman Health and Wellbeing\b",
    'Infected and Ill People': r"\bInfected and Ill People\b",

    'Road Infrastructure': r"\b(Road Infrastructure|road)s?\b",
    'Other Transportation Infrastructure': r"\bOther Transportation Infrastructure\b",
    'Water, Sanitation, and Hygiene Infrastructure': r"\bWater,?\s*Sanitation,?\s*and Hygiene Infrastructure\b",
    'Healthcare Infrastructure': r"\bHealthcare Infrastructure\b",
    'IT and Communication Infrastructure': r"\bIT and Communication Infrastructure\b",

    'Residential Buildings': r"\bResidential Buildings\b",
    'Informal settlements': r"\bInformal settlements\b",
    'Education Infrastructure': r"\bEducation Infrastructure\b",
    'Power and Energy Production Infrastructure': r"\bPower and Energy Production Infrastructure\b",
    'Agriculture Infrastructure': r"\bAgricultur(?:e|al)? Infras(?:tructure|tucture)\b",

    'Crop Production and Forestry': r"\bCrop Production and Forestry\b",
    'Affected Livestock and Animals': r"\bAffected Livestock and Animals\b",

    'Other Economic and Livelihood Impacts': r"\b(Other Economic(?: Activity)? (?:and|&) Livelihood (?:Production|Impact(?:s)?)|Economy and Market|Livelihood|employment|basic needs)\s*\b",
    'Recreation, Tourism, and Culture': r"\bRecreation, Tourism, and Culture\b",

    'Access to Healthcare': r"\bAccess to Healthcare\b",
    'Access to transport and Mobility': r"\bAccess to transport and Mobility\b",
    'Water Quality and Availability': r"\bWater Quality and Availability\b",
    'Access to Education': r"\bAccess to Education\b",
    'Access to Power and Energy': r"\bAccess to Power and Energy\b",
    'Access to Food': r"\bfood\b",
    'Access to Water, Sanitation, and Hygiene': r"\bAccess to Water,?\s*Sanitation,?\s*and Hygiene\b",

    'Other Infrastructure Impacts': r"\bOther Infrastructur(?:e)? Impacts?\b",
    'Other Human Impacts': r"\bOther Human.* Impacts?\b",
    'Other Environmental Impacts': r"\bOther Environmental.* Impacts?\b",
    'Other Service Access Impacts': r"\bOther Service Access.* Impacts?\b",
    'Other Agricultural Impacts': r"\bOther Agricultural.* Impacts?\b",
}
def reclassify_impact_subtype(extracted_data, allowed_impact_types, impact_kw_reclass):
    def reclass_impact_subtype(x):
        if x["impactSubtype"] in allowed_impact_types:
            return x["impactSubtype"]
        candidates = []
        for key, value in impact_kw_reclass.items():
            if re.search(value, x["impactSubtype"], re.IGNORECASE):
                candidates.append(key)
        if len(candidates) == 1:
            return candidates[0]
        else:
            return "Unknown"
    extracted_data["impactSubtype"] = extracted_data.apply(reclass_impact_subtype, axis=1)
    return extracted_data

response_df_proc = reclassify_impact_subtype(response_df_proc, impactSubtype_list, impact_kw_reclass)


In [ ]:
#response_df_proc[["impactSubtype", "impactSubtype_reclass"]].where(response_df_proc["impactSubtype_reclass"]=="Unknown").dropna(how="all")

,impactSubtype,impactSubtype_reclass


In [29]:
#reclassify hazard
hazard_kw_reclass = {
    'Drought': r"\bdrought.*|\bdry\s+spell.*",
    'Wildfire': r"\b(forest\s*fire|wild\s*fire|land\s*fire|bush\s*fire|wildfire|landfire|bushfire|fire)s?\b.*",
    'Earthquake': r"\b(earthquake|ground\s+movement|tsunami)s?\b.*",
    'Mass movement': r"\b(mass\s+movement|avalanche|land\s*slide|landslide|rockfall|sudden\s+subsidence|mudslide|rockslide)s?\b.*",
    'Volcanic activity': r"\b(volcanic|ash\s+fall|lava\s+flow|pyroclastic\s+flow|lahar)s?\b.*",
    'Flood': r"\b(flood|inundation|coastal\s+flood|flash\s+flood|riverine\s+flood|ice\s+jam\s+flood|heavy rain)s?\b.*",
    'Wave action': r"\b(wave|rogue\s+wave|seiche)\b.*",
    'Extreme cold temperature': r"\b(extreme\s+cold\s+temperature|cold\s+wave|coldwave|cold\s+spell|severe\s+winter\s+conditions)s?\b.*",
    'Extreme warm temperature': r"\b(extreme\s+warm\s+temperature|heat\s+wave|heatwave|heat\s+episode|(?:heat|hot)\s+spell|heat\s+stress)s?\b.*",
    'Tropical storm': r"\b(tropical\s+storm|typhoon|hurricane|cyclonic\s+storm)s?\b.*",
    'Other storm': r"\b(extra-?tropical\s+storm|winter\s*storm|storm\s+surge|superstorm|windstorm|snowstorm|blizzard|convective\s+storm|derecho|hail|lightning|tornado|thunderstorm)s?\b.*",
    'Epidemics': r"\b(cholera|dengue|outbreak|epidemic)s?\b.*",
    'Conflict': r"\b(conflict|war|terrorism|unrest)s?\b.*"
}

def reclassify_hazard(extracted_data, hazard_kw_reclass):
    def reclass_haz(x):
        corr_haz = cp.deepcopy(x["hazards"])
        if any([haz for haz in x["hazards"] if haz not in hazard_kw_reclass.keys()]):
            for i, haz in enumerate(x["hazards"]):
                if haz not in hazard_kw_reclass.keys():
                    candidates = [haz_corr for haz_corr in hazard_kw_reclass.keys() if re.search(hazard_kw_reclass[haz_corr], haz, re.IGNORECASE)]
                    if len(candidates) == 1:
                        corr_haz[i] = candidates[0]
                    else:
                        corr_haz[i] = "Unknown"
        return corr_haz
    extracted_data["hazards_reclass"] = extracted_data.apply(reclass_haz, axis=1)
    return extracted_data

response_df_proc = reclassify_hazard(response_df_proc, hazard_kw_reclass)
explode_lists(response_df_proc).hazards_reclass.value_counts()

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


hazards_reclass
Flood                       264
Extreme cold temperature    130
Mass movement                78
Tropical storm               55
Other storm                  45
Drought                      44
Volcanic activity            35
Earthquake                   32
Wildfire                      8
Epidemics                     7
Unknown                       2
Name: count, dtype: int64

In [31]:
wrong_haz = explode_lists(response_df_proc).copy()
wrong_haz = wrong_haz[wrong_haz["hazards_reclass"] == "Unknown"]
wrong_haz["hazards"]

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:89: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  repeat_counts = df[list_columns].applymap(len).max(axis=1)


585    Economic inflation
587    Economic inflation
Name: hazards, dtype: object

In [32]:
impactSubtype_list

['Affected People',
 'Injured People',
 'Displaced People',
 'Homeless People',
 'Missing People',
 'Human Deaths',
 'Human Health and Wellbeing',
 'Infected and Ill People',
 'Road Infrastructure',
 'Other Transportation Infrastructure',
 'Water, Sanitation, and Hygiene Infrastructure',
 'Healthcare Infrastructure',
 'IT and Communication Infrastructure',
 'Residential Buildings',
 'Informal settlements',
 'Education Infrastructure',
 'Power and Energy Production Infrastructure',
 'Agriculture Infrastructure',
 'Crop Production and Forestry',
 'Affected Livestock and Animals',
 'Other Economic and Livelihood Impacts',
 'Recreation, Tourism, and Culture',
 'Access to Healthcare',
 'Access to transport and Mobility',
 'Water Quality and Availability',
 'Access to Education',
 'Access to Power and Energy',
 'Access to Food',
 'Access to Water, Sanitation, and Hygiene',
 'Other Human Impacts',
 'Other Infrastructure Impacts',
 'Other Agricultural Impacts',
 'Other Service Access Impacts']

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people

In [33]:
#reclassify units
unit_converter = {"families" : (3, "people"),
                  "households": (3, "people"),
                  "village": (1000, "people"),
                  "communities": (100, "people"),
                  "USD": (1, "CHF"),
                  "$" : (1, "CHF"),
                  "EUR": (1, "CHF"),
                  "€": (1, "CHF"),
                  }

unit_type_kw_reclass = {#regex actually here should not be necessary due to standardization
                        'km' : r"\b(kilometer|kilometre|km)s?(?!\s*(\*\*\s*2|\^2|²|square|squared|2))",
                        'km**2' : r"\b(kilometer|kilometre|km)s?\s?(\*\*\s*2|\*\*2|\^2|²|square|squared|2)",
                        'kg' : r"(kg.*|.*kilogram.*)",
                        'm**3' : r"\b(meter|metre|m)s?\s?(\*\*\*\s*3|\*\*3|\^3|³|cube|cubic|3)",
                        '%' : r"(%|perc.*)",
}
unit_kw_reclass ={
                        'people': r"people.*|person.*|women.*|men.*|child.*|children.*|adult.*|adults.*|elder.*|elderly.*|infant.*|infants.*|individual.",
                        'roads' : r"road.*|route.*|bridge.*|highway.*|motorway.*",#r"(?<!kilometer|kilometre|km).*(road.*|route.*|.*bridge.*|.*highway.*|.*motorway.*)",
                        'transportation facilities' : r"rail.*|train track.*|airport.*|\scar.*|railway.*|train.*|bus.*|taxi.*|taxicab.*|truck.*",
                        'water, sanitation and hygiene facilities' : r"water.*|sanitation.*|hygiene.*|latrine.*|well.*|tap.*|reservoir.*|aqueduct.*",
                        'healthcare facilities' : r"health|hospital.*|clinic.*|maternity.*|medical",
                        'IT and communication facilities' : r"communication.*|radio.*|tv.*|cell tower.*|antenna.*",
                        'homes' : r"residential.*|residence.|hous.*|home.*|building.*",
                        'education facilities' : r"education.*|school.*|university.*|college.*",
                        'crop production and forestry' : r"crop.*|field.*|forest.*|tree.*|banana.*|coffee.*|cocoa.*|cotton.*|maize.*|rice.*|sorghum.*|soybean.*|sugar.*|tobacco.*|wheat.*",
                        'agricultural facilities' : r"irrigation.*|barn.*|farm.*",
                        'affected animals' : r"livestock.*|animal.*|fish.*|cow.*|sheep.*|poult.*|cattle.*|goat.*|pig.*|chick.*|horse.*|heads?",
                        'informal settlements' : r"camp.?|tent.?|refuge.?|settlement.?"
                         }
default_subtype_unit = {
 'Affected People': "people",
 'Injured People': "people",
 'Displaced People': "people",
 'Homeless People': "people",
 'Missing People': "people",
 'Human Deaths': "people",
 'Residential Buildings': "homes",
 'Informal settlements': "undefined informal settlements",
 'Education Infrastructure': "schools",
 'Human Health and Wellbeing' : "unknown",
 'Infected and Ill People': "people",
 'Road Infrastructure' : "roads",
 'Other Transportation Infrastructure' : "undefined other transportation infrastructure",
 'Water, Sanitation, and Hygiene Infrastructure': "undefined WASH facilities",
 'Healthcare Infrastructure': "undefined healthcare facilities",
 'IT and Communication Infrastructure': "undefined IT and communication facilities",
 'Residential Buildings': "houses",
 'Informal settlements': "undefined informal settlements",
 'Education Infrastructure': "schools",
 'Power and Energy Production Infrastructure' : "undefined power and energy production infrastructure facilities",
 'Agriculture Infrastructure': "undefined agricultural facilities",
 'Crop Production and Forestry': "undefined crop production and forestry",
 'Affected Livestock and Animals': "undefined affected animals",
 'Other Economic and Livelihood Impacts': "CHF",
 #'Water Quality and Availability':
 'Recreation, Tourism, and Culture' : "unknown",
 'Access to Healthcare': "people",
 'Access to transport and Mobility': "people",
 'Water Quality and Availability' : "unknown",
 'Access to Education':"people",
 'Access to Power and Energy':"people",
 'Access to Food':"people",
 'Access to Water, Sanitation, and Hygiene':"people",
 'Other Human Impacts': "unknown",
 'Other Infrastructure Impacts': "unknown",
 'Other Agricultural Impacts': "unknown",
 'Other Service Access Impacts': "people"
}

def convert_unit(extracted_data, unit_converter):
    """Convert units that can be converted e.g. families => people"""
    def convert(x):
        unit = x['impactUnit']
        if not isinstance(unit, str):
            return x  # skip if unit is None or not a string

        unit = unit.strip()
        if unit == "":
            return x

        for old_unit, (conv_fact, new_unit) in unit_converter.items():
            try:
                if unit == old_unit:
                    x["impactValue"] = float(x["impactValue"]) #force conversion to float
                    x["impactValue"] = conv_fact*x["impactValue"]
                    x["impactUnit"] = new_unit
            except Exception as e:
                print(f"Skipping unit conversion for row due to error: {e}")
                continue
        return x
    extracted_data = extracted_data.apply(convert, axis=1)
    return extracted_data


def assign_unit_type(extracted_data, unit_type_kw_reclass):
    """Detect if dimension of unit can be identified e.g. length, mass,...
       Default to "other"
    """
    def assign_type(x):
        unit = str(x["impactUnit"]).lower() #ensure unit is string
        candidates = [unit_type for unit_type in unit_type_kw_reclass.keys() if re.search(unit_type_kw_reclass[unit_type], unit, re.IGNORECASE)]
        if len(candidates) == 1:
            return candidates[0]
        elif len(candidates) == 0:
            return "other"
        else:
            return "multiple"
    extracted_data["unit_type"] = extracted_data.apply(assign_type, axis=1)
    return extracted_data

def reclassify_units(extracted_data, unit_kw_reclass, default_subtype_unit):
    def reclass_units(x):
        unit = str(x["impactUnit"]).lower() #ensure unit is string
        unit_type = x['unit_type']
        unit_prefix = f"{unit_type} of " if unit_type != "other" else ""
        candidates = [unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]
        if len(candidates) == 1:
            return unit_prefix+candidates[0]
        else:
            #no unit identified, infer unit from category
            inferred_unit = default_subtype_unit[x["impactSubtype"]] if x["impactSubtype"] != "Unknown" else unit
            return unit_prefix+inferred_unit
    extracted_data["impactUnit"] = extracted_data.apply(reclass_units, axis=1)
    return extracted_data

from src.text_processing_functions import *
def standardize_units(value, unit):
    """Standardize units to a common baseline in text"""

    ureg = UnitRegistry()
    #print(f"{value} {unit}")
    identified_units = []
    identified_patterns = []
    for target_unit, unit_patterns in std_unit_kw_reclass.items():
        for pattern in unit_patterns:
            if re.search(pattern, unit, re.IGNORECASE):
                identified_units.append(target_unit)
                identified_patterns.append(pattern)
    #matched = [(target_unit, unit_patterns) for target_unit, unit_patterns in std_unit_kw_reclass.items() if np.any([re.search(pattern, unit, re.IGNORECASE) for pattern in unit_patterns])]
    if len(identified_units) == 0:
        return pd.Series({"impactValue": value, "impactUnit": unit})
    elif len(identified_units) > 1:
        raise ValueError(f"Multiple potential units found for token: {unit}")
    identified_unit = identified_units[0]
    identified_pattern = identified_patterns[0]
    si_unit = unit_mapping[identified_unit]
    # Perform conversion
    quantity = float(value) * ureg(identified_unit)
    converted_quantity = quantity.to(si_unit)
    converted_value = converted_quantity.magnitude
    converted_unit = re.sub(identified_pattern, si_unit, unit)

    return pd.Series({"impactValue": converted_value, "impactUnit": converted_unit})

def standardize_value_units(response_df):
    def join_value_units(x):
        return str(x["impactValue"]) +  "," + str(x["impactUnit"])
    def split_value_units(x):
        return x["value_unit"].split(",")
    def apply_std_units(x):
        return standardize_units(str(x["impactValue"]), str(x["impactUnit"]))
    #response_df["value_unit"] = response_df.apply(join_value_units, axis=1)
    #response_df["value_unit"] = response_df.apply(split_value_units, axis=1)
    response_df[["impactValue", "impactUnit"]]  = response_df.apply(apply_std_units, axis=1)
    return response_df



In [34]:
unit_type = "kg"
unit = "kg of crops"
[unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]


['crop production and forestry']

In [35]:
test_df = pd.DataFrame({
        "impactSubtype": ["Affected People", "Crop Production and Forestry", "Crop Production and Forestry"],
        "impactValue": [1000,1000,100],
        "impactUnit": ["families", "kg of crops","hectares of crops"]})
test_df = standardize_value_units(test_df)
test_df = convert_unit(test_df, unit_converter)
test_df = assign_unit_type(test_df, unit_type_kw_reclass)
test_df = reclassify_units(test_df, unit_kw_reclass, default_subtype_unit)
test_df

,impactSubtype,impactValue,impactUnit,unit_type
0,Affected People,3000.0,people,other
1,Crop Production and Forestry,1000.0,kg of crop production and forestry,kg
2,Crop Production and Forestry,1.0,km**2 of crop production and forestry,km**2


In [36]:
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
response_df_proc = standardize_value_units(response_df_proc)
response_df_proc = convert_unit(response_df_proc, unit_converter)
response_df_proc = assign_unit_type(response_df_proc, unit_type_kw_reclass)
response_df_proc = reclassify_units(response_df_proc, unit_kw_reclass, default_subtype_unit)

In [37]:
response_df_proc[["impactSubtype","impactValueOrig", "impactValue", "impactUnitOrig", "impactUnit","impactsAnnotation"]]

,impactSubtype,impactValueOrig,impactValue,impactUnitOrig,impactUnit,impactsAnnotation
0,Affected People,247408.0,247408.0,people,people,"[number of people affected:247,408]"
1,Human Health and Wellbeing,nan,nan,unknown,unknown,"[In Vanuatu, 70% of rural areas and 40% of urb..."
2,Residential Buildings,nan,nan,houses,homes,"[In Vanuatu, 70% of rural areas and 40% of urb..."
3,Other Transportation Infrastructure,nan,nan,undefined other transportation infrastructure,undefined other transportation infrastructure,[The complex logistics and expense involved in...
4,"Water, Sanitation, and Hygiene Infrastructure",68.0,68.0,% of undefined WASH facilities,% of undefined WASH facilities,"[In Vanuatu, it is estimated that 68 per cent ..."
...,...,...,...,...,...,...
340,Crop Production and Forestry,nan,nan,undefined crop production and forestry,crop production and forestry,[zambia is undergoing one the driest agricultu...
341,Affected Livestock and Animals,nan,nan,undefined affected animals,affected animals,[zambia is undergoing one the driest agricultu...
342,Access to Healthcare,nan,nan,people,people,[the decreased access to water has also led to...
343,Access to Education,nan,nan,people,people,[many households are struggling to meet their ...


In [38]:
# save
savename = "post_processed_" + res_savename
response_df_proc.to_csv(DATA_OUT_PROC + savename, index=False)

In [39]:
response_df_proc[["impactValue", "impactUnit", "impactValueOrig", "impactUnitOrig", "impactsAnnotation"]]

,impactValue,impactUnit,impactValueOrig,impactUnitOrig,impactsAnnotation
0,247408.0,people,247408.0,people,"[number of people affected:247,408]"
1,nan,unknown,nan,unknown,"[In Vanuatu, 70% of rural areas and 40% of urb..."
2,nan,homes,nan,houses,"[In Vanuatu, 70% of rural areas and 40% of urb..."
3,nan,undefined other transportation infrastructure,nan,undefined other transportation infrastructure,[The complex logistics and expense involved in...
4,68.0,% of undefined WASH facilities,68.0,% of undefined WASH facilities,"[In Vanuatu, it is estimated that 68 per cent ..."
...,...,...,...,...,...
340,nan,crop production and forestry,nan,undefined crop production and forestry,[zambia is undergoing one the driest agricultu...
341,nan,affected animals,nan,undefined affected animals,[zambia is undergoing one the driest agricultu...
342,nan,people,nan,people,[the decreased access to water has also led to...
343,nan,people,nan,people,[many households are struggling to meet their ...
